# Feature Exploration: High Wind Days

This notebook tests the impact of adding high wind days as a feature. Adding weather features in notebooks 4c, 4d and 4e has improved the performance of the model. The next highest correlated weather feature in notebook 3b is the number of high wind days. It is also very intuitive that high wind days (>8 m/s) may disrupt flight operations. Here, it is defined as:

`days_high_wind_total` = days_high_wind_dep + days_high_wind_arr

As with other weather features, the exponential transformation, `days_high_wind_total_exp`, will also be tested.


After observing their positive impact in the previous notebooks, the following features are included in the baseline model:
- From 4c: `rainy_days_arr_exp` (regression), `rainy_days_arr` (classification)
- From 4d: `delay_rate_gradient` (momentum)
- From 4e: `temp_volatility_total_exp` (temperature volatility)

**Models tested:**
- Regression: Ridge, Random Forest
- Classification: XGBoost, Random Forest, Logistic Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed (pip install xgboost)")

%matplotlib inline

## 1. Data Preparation

In [ ]:
# Load data with weather features
df = pd.read_csv('../data/processed/ml_training_data_syd_mel_with_holidays.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = pd.to_datetime(df['month']).dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']

# Sort for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

In [ ]:
# Create lagged features (from 4d)
df['delay_rate_lag1'] = df.groupby('airline_route')['delay_rate'].shift(1)
df['delay_rate_lag2'] = df.groupby('airline_route')['delay_rate'].shift(2)

# Create momentum feature: gradient from lag2 to lag1 (from 4d)
df['delay_rate_gradient'] = df['delay_rate_lag1'] - df['delay_rate_lag2']

# Create cyclical month encoding
df['month_sin'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df['airline'], prefix='airline')
df = pd.concat([df, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

# Create exponential transformation of rainy_days_arr (from 4c - for regression)
df['rainy_days_arr_exp'] = np.exp(df['rainy_days_arr'] / df['rainy_days_arr'].max())

# Create temperature volatility total (from 4e)
df['temp_volatility_total'] = df['temp_volatility_dep'] + df['temp_volatility_arr']
df['temp_volatility_total_exp'] = np.exp(df['temp_volatility_total'] / df['temp_volatility_total'].max())

# Create new feature: high wind days total (dep + arr)
df['days_high_wind_total'] = df['days_high_wind_dep'] + df['days_high_wind_arr']

# Create exponential transformation of days_high_wind_total
df['days_high_wind_total_exp'] = np.exp(df['days_high_wind_total'] / df['days_high_wind_total'].max())

print(f"days_high_wind_total range: {df['days_high_wind_total'].min():.2f} - {df['days_high_wind_total'].max():.2f}")
print(f"days_high_wind_total_exp range: {df['days_high_wind_total_exp'].min():.2f} - {df['days_high_wind_total_exp'].max():.2f}")

# Drop rows with missing lag values (need lag2 for gradient)
df_clean = df.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"Rows after dropping NaN: {len(df_clean)}")

### Train/Validation/Test Split

Split the training data:
* Train: 2010-2017 and 2024 for actual model training
* Validation: 2018 and 2023 for cross-validation (tuning hyperparameters)
* Test: 2019 and 2025 for final evaluation

It is very important to stratify the train-test split so that each split contains both pre and post-COVID years.

In [ ]:
# Time-based split (excluding 2020-2022 COVID period)
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")
print("\nNote: 2020-2022 excluded (COVID period)")

In [ ]:
# Define feature sets
base_features = airline_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

# Regression: rainy_days_arr_exp + gradient + temp_volatility_total_exp (from 4e)
reg_features_baseline = base_features + ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp']
reg_features_plain = reg_features_baseline + ['days_high_wind_total']
reg_features_exp = reg_features_baseline + ['days_high_wind_total_exp']

# Classification: rainy_days_arr (plain) + gradient + temp_volatility_total_exp (from 4e)
clf_features_baseline = base_features + ['rainy_days_arr', 'delay_rate_gradient', 'temp_volatility_total_exp']
clf_features_plain = clf_features_baseline + ['days_high_wind_total']
clf_features_exp = clf_features_baseline + ['days_high_wind_total_exp']

print("Regression features:")
print(f"  Baseline (4e model):      {len(reg_features_baseline)} features")
print(f"  + plain days_high_wind:   {len(reg_features_plain)} features")
print(f"  + exp days_high_wind:     {len(reg_features_exp)} features")
print("\nClassification features:")
print(f"  Baseline (4e model):      {len(clf_features_baseline)} features")
print(f"  + plain days_high_wind:   {len(clf_features_plain)} features")
print(f"  + exp days_high_wind:     {len(clf_features_exp)} features")

In [ ]:
# Prepare target variables
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")

## 2. Regression Models

In [ ]:
# Train regression models: baseline vs plain vs exp days_high_wind
reg_results = []

feature_variants = [
    ('baseline', reg_features_baseline),
    ('plain_wind', reg_features_plain),
    ('exp_wind', reg_features_exp)
]

for variant_name, features in feature_variants:
    print(f"\n{'='*60}")
    print(f"Regression with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Ridge Regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train_reg)
    ridge_test_pred = ridge.predict(X_test_scaled)
    
    ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
    print(f"  Ridge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}")
    
    reg_results.append({
        'Variant': variant_name,
        'Model': 'Ridge',
        'Test_R2': ridge_r2,
        'Test_RMSE': ridge_rmse
    })
    
    # Random Forest Regression
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train, y_train_reg)
    rf_test_pred = rf_reg.predict(X_test)
    
    rf_r2 = r2_score(y_test_reg, rf_test_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
    print(f"  RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}")
    
    reg_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Test_R2': rf_r2,
        'Test_RMSE': rf_rmse
    })

reg_df = pd.DataFrame(reg_results)
print("\nRegression models trained.")

## 3. Classification Models

In [ ]:
# Train classification models: baseline vs plain vs exp days_high_wind
clf_results = []

feature_variants = [
    ('baseline', clf_features_baseline),
    ('plain_wind', clf_features_plain),
    ('exp_wind', clf_features_exp)
]

for variant_name, features in feature_variants:
    print(f"\n{'='*60}")
    print(f"Classification with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Logistic Regression
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_test_pred = logreg.predict(X_test_scaled)
    logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
    
    logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
    logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
    print(f"  Logistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}")
    
    clf_results.append({
        'Variant': variant_name,
        'Model': 'Logistic',
        'Test_F1': logreg_f1,
        'Test_AUC': logreg_auc
    })
    
    # Random Forest Classification
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_test_pred = rf_clf.predict(X_test)
    rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
    
    rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
    rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
    print(f"  RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}")
    
    clf_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Test_F1': rf_clf_f1,
        'Test_AUC': rf_clf_auc
    })
    
    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            min_child_weight=5, random_state=42, n_jobs=-1
        )
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_test_pred = xgb_clf.predict(X_test)
        xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
        
        xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
        xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
        print(f"  XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}")
        
        clf_results.append({
            'Variant': variant_name,
            'Model': 'XGBoost',
            'Test_F1': xgb_f1,
            'Test_AUC': xgb_auc
        })

clf_df = pd.DataFrame(clf_results)
print("\nClassification models trained.")

## 4. Side-by-Side Comparison

In [ ]:
# Regression comparison: baseline vs plain vs exp
print("=" * 100)
print("REGRESSION: baseline (4e) vs plain_wind vs exp_wind")
print("=" * 100)

reg_pivot = reg_df.pivot(index='Model', columns='Variant', values=['Test_R2', 'Test_RMSE'])

# Flatten column names
reg_pivot.columns = [f'{col[1]}_{col[0]}' for col in reg_pivot.columns]
reg_pivot = reg_pivot.reset_index()

# Calculate differences from baseline
reg_pivot['plain_R2_diff'] = reg_pivot['plain_wind_Test_R2'] - reg_pivot['baseline_Test_R2']
reg_pivot['exp_R2_diff'] = reg_pivot['exp_wind_Test_R2'] - reg_pivot['baseline_Test_R2']

print(f"\n{'Model':<15} {'base R²':>10} {'plain R²':>10} {'exp R²':>10} {'plain diff':>12} {'exp diff':>10}")
print("-" * 80)
for _, row in reg_pivot.iterrows():
    plain_sign = '+' if row['plain_R2_diff'] > 0 else ''
    exp_sign = '+' if row['exp_R2_diff'] > 0 else ''
    print(f"{row['Model']:<15} {row['baseline_Test_R2']:>10.4f} {row['plain_wind_Test_R2']:>10.4f} {row['exp_wind_Test_R2']:>10.4f} {plain_sign}{row['plain_R2_diff']:>11.4f} {exp_sign}{row['exp_R2_diff']:>9.4f}")

In [ ]:
# Classification comparison: baseline vs plain vs exp
print("=" * 100)
print("CLASSIFICATION: baseline (4e) vs plain_wind vs exp_wind")
print("=" * 100)

clf_pivot = clf_df.pivot(index='Model', columns='Variant', values=['Test_F1', 'Test_AUC'])

# Flatten column names
clf_pivot.columns = [f'{col[1]}_{col[0]}' for col in clf_pivot.columns]
clf_pivot = clf_pivot.reset_index()

# Calculate differences from baseline
clf_pivot['plain_F1_diff'] = clf_pivot['plain_wind_Test_F1'] - clf_pivot['baseline_Test_F1']
clf_pivot['exp_F1_diff'] = clf_pivot['exp_wind_Test_F1'] - clf_pivot['baseline_Test_F1']

print(f"\n{'Model':<15} {'base F1':>10} {'plain F1':>10} {'exp F1':>10} {'plain diff':>12} {'exp diff':>10}")
print("-" * 80)
for _, row in clf_pivot.iterrows():
    plain_sign = '+' if row['plain_F1_diff'] > 0 else ''
    exp_sign = '+' if row['exp_F1_diff'] > 0 else ''
    print(f"{row['Model']:<15} {row['baseline_Test_F1']:>10.4f} {row['plain_wind_Test_F1']:>10.4f} {row['exp_wind_Test_F1']:>10.4f} {plain_sign}{row['plain_F1_diff']:>11.4f} {exp_sign}{row['exp_F1_diff']:>9.4f}")

In [ ]:
# Visualization: Side-by-side bar charts for three variants
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Regression R²
ax = axes[0, 0]
x = np.arange(len(reg_pivot))
width = 0.25
ax.bar(x - width, reg_pivot['baseline_Test_R2'], width, label='baseline (4e)', color='steelblue', alpha=0.8)
ax.bar(x, reg_pivot['plain_wind_Test_R2'], width, label='+ plain wind', color='#27ae60', alpha=0.8)
ax.bar(x + width, reg_pivot['exp_wind_Test_R2'], width, label='+ exp wind', color='#e67e22', alpha=0.8)
ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(reg_pivot['Model'])
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)

# Regression RMSE
ax = axes[0, 1]
ax.bar(x - width, reg_pivot['baseline_Test_RMSE'], width, label='baseline (4e)', color='steelblue', alpha=0.8)
ax.bar(x, reg_pivot['plain_wind_Test_RMSE'], width, label='+ plain wind', color='#27ae60', alpha=0.8)
ax.bar(x + width, reg_pivot['exp_wind_Test_RMSE'], width, label='+ exp wind', color='#e67e22', alpha=0.8)
ax.set_ylabel(r'Test RMSE')
ax.set_title(r'Regression: RMSE Comparison (lower is better)')
ax.set_xticks(x)
ax.set_xticklabels(reg_pivot['Model'])
ax.legend()

# Classification F1
ax = axes[1, 0]
x = np.arange(len(clf_pivot))
ax.bar(x - width, clf_pivot['baseline_Test_F1'], width, label='baseline (4e)', color='steelblue', alpha=0.8)
ax.bar(x, clf_pivot['plain_wind_Test_F1'], width, label='+ plain wind', color='#27ae60', alpha=0.8)
ax.bar(x + width, clf_pivot['exp_wind_Test_F1'], width, label='+ exp wind', color='#e67e22', alpha=0.8)
ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(clf_pivot['Model'])
ax.legend()
ax.set_ylim(0, 1)

# Classification AUC
ax = axes[1, 1]
ax.bar(x - width, clf_pivot['baseline_Test_AUC'], width, label='baseline (4e)', color='steelblue', alpha=0.8)
ax.bar(x, clf_pivot['plain_wind_Test_AUC'], width, label='+ plain wind', color='#27ae60', alpha=0.8)
ax.bar(x + width, clf_pivot['exp_wind_Test_AUC'], width, label='+ exp wind', color='#e67e22', alpha=0.8)
ax.set_ylabel(r'Test AUC')
ax.set_title(r'Classification: AUC Comparison')
ax.set_xticks(x)
ax.set_xticklabels(clf_pivot['Model'])
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 5. Feature Importance Analysis

In [ ]:
# Train final RF model with exp days_high_wind to check feature importance
X_train_reg = df_clean.loc[train_mask, reg_features_exp].values
rf_final = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_final.fit(X_train_reg, y_train_reg)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': reg_features_exp,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF Regression with exp days_high_wind):")
print("-" * 50)
for _, row in importance_df.head(12).iterrows():
    print(f"  {row['Feature']:<25} {row['Importance']:.4f}")

# Highlight new feature
wind_importance = importance_df[importance_df['Feature'] == 'days_high_wind_total_exp']['Importance'].values[0]
wind_rank = list(importance_df['Feature']).index('days_high_wind_total_exp') + 1
print(f"\ndays_high_wind_total_exp rank: {wind_rank} (importance: {wind_importance:.4f})")

In [ ]:
# Visualize top features
fig, ax = plt.subplots(figsize=(10, 6))

top_features = importance_df.head(12)
colors = ['#e67e22' if 'wind' in f else 
          '#9b59b6' if 'temp_volatility' in f else
          '#27ae60' if f == 'delay_rate_gradient' else
          '#3498db' if 'rainy' in f else 'steelblue' 
          for f in top_features['Feature']]

ax.barh(top_features['Feature'], top_features['Importance'], color=colors, alpha=0.8)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 12 Feature Importances (RF Regression)\n(orange=wind, purple=temp_vol, green=momentum, blue=rain)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

### 5.1 Time-Series: Actual vs Predicted (Ridge Regression with exp days_high_wind)

In [ ]:
# Time-series comparison: Actual vs Predicted delay_rate (Ridge with exp days_high_wind)
# Train on train set, predict on FULL dataset to see overall fit

X_train_ridge = df_clean.loc[train_mask, reg_features_exp].values
X_all = df_clean[reg_features_exp].values

scaler_ridge = StandardScaler()
X_train_ridge_scaled = scaler_ridge.fit_transform(X_train_ridge)
X_all_scaled = scaler_ridge.transform(X_all)

ridge_exp = Ridge(alpha=1.0)
ridge_exp.fit(X_train_ridge_scaled, y_train_reg)
all_predictions = ridge_exp.predict(X_all_scaled)

# Create dataframe with all predictions - keep df_clean's index for proper mask alignment
full_df = df_clean[['year_month_dt', 'year', 'airline', 'departing_port', 'arriving_port', 'delay_rate']].copy()
full_df['predicted'] = all_predictions

# Mark train/val/test splits BEFORE sorting (masks are aligned with df_clean's index)
full_df['split'] = 'excluded'  # COVID period
full_df.loc[train_mask, 'split'] = 'train'
full_df.loc[val_mask, 'split'] = 'val'
full_df.loc[test_mask, 'split'] = 'test'

# Now sort for time-series plotting
full_df = full_df.sort_values('year_month_dt')

# Aggregate by month (average across airlines and routes)
monthly_agg = full_df.groupby('year_month_dt').agg({
    'delay_rate': 'mean',
    'predicted': 'mean',
    'split': 'first'
}).reset_index()

# Plot time-series comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Full time series with train/val/test regions highlighted
ax = axes[0]

# Plot actual and predicted
ax.plot(monthly_agg['year_month_dt'], monthly_agg['delay_rate'], 'o-', 
        label='Actual', color='steelblue', linewidth=1.5, markersize=4, alpha=0.8)
ax.plot(monthly_agg['year_month_dt'], monthly_agg['predicted'], 's-', 
        label='Predicted (Ridge + exp wind)', color='#e67e22', linewidth=1.5, markersize=4, alpha=0.8)

# Shade regions by split
train_months = monthly_agg[monthly_agg['split'] == 'train']['year_month_dt']
val_months = monthly_agg[monthly_agg['split'] == 'val']['year_month_dt']
test_months = monthly_agg[monthly_agg['split'] == 'test']['year_month_dt']
excluded_months = monthly_agg[monthly_agg['split'] == 'excluded']['year_month_dt']

if len(excluded_months) > 0:
    ax.axvspan(excluded_months.min(), excluded_months.max(), alpha=0.15, color='gray', label='COVID excluded')

ax.axhline(0.25, color='orange', linestyle=':', linewidth=1.5, label='High delay threshold (25%)')
ax.set_xlabel('Month')
ax.set_ylabel('Delay Rate')
ax.set_title('Full Dataset: Actual vs Predicted Delay Rate (Monthly Average, All Airlines)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

# Plot 2: Scatter plot of actual vs predicted by split
ax = axes[1]

split_colors = {'train': 'steelblue', 'val': '#27ae60', 'test': '#e67e22', 'excluded': 'gray'}
split_labels = {'train': 'Train (2010-17, 2024)', 'val': 'Val (2018, 2023)', 
                'test': 'Test (2019, 2025)', 'excluded': 'COVID (2020-22)'}

for split_name in ['train', 'val', 'test', 'excluded']:
    split_data = full_df[full_df['split'] == split_name]
    if len(split_data) > 0:
        ax.scatter(split_data['delay_rate'], split_data['predicted'], 
                   alpha=0.4, s=20, c=split_colors[split_name], label=split_labels[split_name])

# Perfect prediction line
ax.plot([0, 0.8], [0, 0.8], 'k--', linewidth=1, label='Perfect prediction')
ax.set_xlabel('Actual Delay Rate')
ax.set_ylabel('Predicted Delay Rate')
ax.set_title('Actual vs Predicted: All Data Points by Split')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(0, 0.8)
ax.set_ylim(0, 0.8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics by split using sklearn's r2_score for consistency
print("\nPrediction Summary by Split (Ridge with exp days_high_wind):")
print("-" * 60)
for split_name in ['train', 'val', 'test', 'excluded']:
    split_data = full_df[full_df['split'] == split_name]
    if len(split_data) > 0:
        mae = np.abs(split_data['delay_rate'] - split_data['predicted']).mean()
        corr = split_data['delay_rate'].corr(split_data['predicted'])
        r2 = r2_score(split_data['delay_rate'], split_data['predicted'])
        print(f"  {split_labels[split_name]:<25} MAE={mae:.4f}, R²={r2:.4f}, Corr={corr:.4f}")

## 6. Summary and Observations

In [ ]:
# Summary
print("=" * 80)
print("SUMMARY: Impact of days_high_wind_total (plain vs exp)")
print("=" * 80)
print("\nBaseline (4e) includes: rainy_days + delay_rate_gradient + temp_volatility_total_exp")

# Reference values from 4e (with exp temp_volatility)
ref_4e = {
    'Ridge': {'R2': 0.4941, 'RMSE': 0.0703},
    'Random Forest': {'R2': 0.4715, 'RMSE': 0.0719},
    'Logistic': {'F1': 0.7746, 'AUC': 0.8852},
    'Random Forest_clf': {'F1': 0.7514, 'AUC': 0.8555},
    'XGBoost': {'F1': 0.8065, 'AUC': 0.8742}
}

print("\nREGRESSION MODELS:")
print("-" * 80)
print(f"{'Model':<20} {'4e R²':>10} {'4f best R²':>12} {'Δ from 4e':>12} {'Best variant':>15}")
print("-" * 80)
for _, row in reg_pivot.iterrows():
    model = row['Model']
    ref_r2 = ref_4e[model]['R2']
    
    plain_r2 = row['plain_wind_Test_R2']
    exp_r2 = row['exp_wind_Test_R2']
    baseline_r2 = row['baseline_Test_R2']
    
    plain_diff = row['plain_R2_diff']
    exp_diff = row['exp_R2_diff']
    
    # Find the best performing variant (including baseline)
    all_variants = {'baseline': baseline_r2, 'plain': plain_r2, 'exp': exp_r2}
    best_variant = max(all_variants, key=all_variants.get)
    best_r2 = all_variants[best_variant]
    
    # Determine if improvement is significant
    if best_variant == 'baseline' or max(plain_diff, exp_diff) <= 0.005:
        best = "neither"
    elif exp_diff > plain_diff:
        best = "exp"
    else:
        best = "plain"
    
    diff_from_4e = best_r2 - ref_r2
    diff_sign = '+' if diff_from_4e > 0 else ''
    print(f"{model:<20} {ref_r2:>10.4f} {best_r2:>12.4f} {diff_sign}{diff_from_4e:>11.4f} {best:>15}")

print("\nCLASSIFICATION MODELS:")
print("-" * 80)
print(f"{'Model':<20} {'4e F1':>10} {'4f best F1':>12} {'Δ from 4e':>12} {'Best variant':>15}")
print("-" * 80)
for _, row in clf_pivot.iterrows():
    model = row['Model']
    if model == 'Random Forest':
        ref_f1 = ref_4e['Random Forest_clf']['F1']
    else:
        ref_f1 = ref_4e[model]['F1']
    
    plain_f1 = row['plain_wind_Test_F1']
    exp_f1 = row['exp_wind_Test_F1']
    baseline_f1 = row['baseline_Test_F1']
    
    plain_diff = row['plain_F1_diff']
    exp_diff = row['exp_F1_diff']
    
    # Find the best performing variant (including baseline)
    all_variants = {'baseline': baseline_f1, 'plain': plain_f1, 'exp': exp_f1}
    best_variant = max(all_variants, key=all_variants.get)
    best_f1 = all_variants[best_variant]
    
    # Determine if improvement is significant
    if best_variant == 'baseline' or max(plain_diff, exp_diff) <= 0.005:
        best = "neither"
    elif exp_diff > plain_diff:
        best = "exp"
    else:
        best = "plain"
    
    diff_from_4e = best_f1 - ref_f1
    diff_sign = '+' if diff_from_4e > 0 else ''
    print(f"{model:<20} {ref_f1:>10.4f} {best_f1:>12.4f} {diff_sign}{diff_from_4e:>11.4f} {best:>15}")

# Count wins for each variant
reg_plain_wins = sum((reg_pivot['plain_R2_diff'] > reg_pivot['exp_R2_diff']) & (reg_pivot['plain_R2_diff'] > 0.005))
reg_exp_wins = sum((reg_pivot['exp_R2_diff'] > reg_pivot['plain_R2_diff']) & (reg_pivot['exp_R2_diff'] > 0.005))
clf_plain_wins = sum((clf_pivot['plain_F1_diff'] > clf_pivot['exp_F1_diff']) & (clf_pivot['plain_F1_diff'] > 0.005))
clf_exp_wins = sum((clf_pivot['exp_F1_diff'] > clf_pivot['plain_F1_diff']) & (clf_pivot['exp_F1_diff'] > 0.005))

total_plain_wins = reg_plain_wins + clf_plain_wins
total_exp_wins = reg_exp_wins + clf_exp_wins
total_models = len(reg_pivot) + len(clf_pivot)

# Comparison with 4e
print("\n" + "=" * 80)
print("COMPARISON WITH 4e (improvement over temp_volatility baseline)")
print("=" * 80)

# Determine best variant for each model
print("\nRegression (R² improvement from 4e → 4f):")
for _, row in reg_pivot.iterrows():
    model = row['Model']
    ref_r2 = ref_4e[model]['R2']
    
    plain_r2 = row['plain_wind_Test_R2']
    exp_r2 = row['exp_wind_Test_R2']
    best_r2 = max(plain_r2, exp_r2)
    best_variant = "exp" if exp_r2 >= plain_r2 else "plain"
    
    diff = best_r2 - ref_r2
    sign = '+' if diff > 0 else ''
    status = "IMPROVED" if diff > 0.005 else ("WORSE" if diff < -0.005 else "~same")
    print(f"  {model:<20}: {ref_r2:.4f} → {best_r2:.4f} ({sign}{diff:.4f}) [{best_variant}] {status}")

print("\nClassification (F1 improvement from 4e → 4f):")
for _, row in clf_pivot.iterrows():
    model = row['Model']
    if model == 'Random Forest':
        ref_f1 = ref_4e['Random Forest_clf']['F1']
    else:
        ref_f1 = ref_4e[model]['F1']
    
    plain_f1 = row['plain_wind_Test_F1']
    exp_f1 = row['exp_wind_Test_F1']
    best_f1 = max(plain_f1, exp_f1)
    best_variant = "exp" if exp_f1 >= plain_f1 else "plain"
    
    diff = best_f1 - ref_f1
    sign = '+' if diff > 0 else ''
    status = "IMPROVED" if diff > 0.005 else ("WORSE" if diff < -0.005 else "~same")
    print(f"  {model:<20}: {ref_f1:.4f} → {best_f1:.4f} ({sign}{diff:.4f}) [{best_variant}] {status}")

### Observations

Adding `days_high_wind_total` has negligible impact to the model performance compared the baseline model in 4e.

## 7. Next Step

It appears that adding more weather features does not add further meaningful improvements. The next step will be to scale the ML training data preparation to take in multiple city pairs. It builds on the single-pair code in notebook 3.